# Phase 0 + 1: Why Math is Hard for AI, and Chain-of-Thought Prompting

**Competition:** AI Mathematical Olympiad — Progress Prize 3  
**Goal of this notebook:** Understand *why* pure LLM generation fails at math, and measure how Chain-of-Thought (CoT) prompting improves it — using the **official AIMO3 Reference Bench** (10 real problems with verified solutions).

---

## What are we trying to do?

The competition asks an AI to solve 110 olympiad-level math problems and return a 5-digit integer answer for each. The problems arrive **one at a time** through a Kaggle evaluation gateway. Here's the full system architecture:

```
Kaggle Gateway (on Kaggle servers)         Your Submission Notebook
──────────────────────────────────         ─────────────────────────
sends 1 problem at a time                  predict(id, problem_text)
  ─── gRPC message ──────────────────>         run model / TIR loop
  <── integer answer ─────────────────         return integer answer

9-hour total time budget                   model must be loaded BEFORE serve()
```

The three files in the Dataset folder:
- **`test.csv`** — 3 dummy problems (format only; real test is on Kaggle)  
- **`reference.csv`** — 10 real problems with verified answers (your local benchmark)  
- **`sample_submission.csv`** — shows the exact `id,answer` output format  

**In this notebook** we use `reference.csv` as our benchmark. It's the closest thing to the actual competition problems you have locally.

---

## Phase 0 — Why is Math Hard for LLMs?

### What an LLM actually does

An LLM (Large Language Model) is, at its core, a **next-token predictor**. Given the sequence of tokens so far, it outputs a probability distribution over what token comes next. It picks (or samples) from that distribution and repeats.

This is a powerful capability — it means the model has absorbed enormous amounts of human knowledge from text. But it has a fatal flaw for math:

> **The model generates what *looks plausible*, not what is *logically correct*.**

In natural language, plausible ≈ correct most of the time. In mathematics, plausible ≠ correct — even a single arithmetic error cascades into a wrong answer.

### The three failure modes

**1. Arithmetic hallucination**  
The model "knows" that large multiplications produce large numbers, but it has no calculator. It generates a number that *looks right in context* but is wrong. Example:
```
37 × 84 = 3108   ← model output (wrong; correct is 3108... wait, actually 3108 is right!)
37 × 84 = 3208   ← but this is what models often say
```
Arithmetic is brittle because the training data has far more examples of small multiplications than large ones.

**2. No working memory**  
Math proofs require holding intermediate results in mind across many steps. The model's "memory" is just the context window — it sees tokens, not variables. It can lose track of which variable equals what.

**3. No ability to verify**  
A human can check: *"Does my answer satisfy the original equation?"* A pure text-generating LLM generates a final answer token and stops. It cannot plug the answer back in and check — unless you explicitly teach it to.

### Why code execution is the key unlock

If the model writes **Python code** and we **execute** it:
- Python's arithmetic is exact — no hallucination
- Variables are stored in memory — no tracking errors  
- We can verify by running the check as code
- `sympy` can do symbolic algebra, calculus, number theory

This is the fundamental insight behind Tool-Integrated Reasoning (TIR), which we'll build in notebook 2. For now, we start simpler.

---

## Phase 1 — Chain-of-Thought (CoT) Prompting

### What is CoT?

Chain-of-Thought prompting is the idea that if you instruct the model to *show its reasoning steps* before giving the final answer, it performs much better on complex tasks.

The canonical zero-shot trigger: **"Let's think step by step."**

**Why does it work?**  
By generating intermediate steps, the model conditions its next-token prediction on those steps. Each reasoning step constrains the space of plausible next tokens, steering away from shortcuts and toward logical deduction. In other words: writing the steps out is how the model "thinks".

**Without CoT:**
```
Q: If 5 machines make 5 widgets in 5 minutes, how many minutes does it take 
   100 machines to make 100 widgets?
A: 100 minutes   ← wrong (common intuitive trap)
```

**With CoT:**
```
Q: ... Let's think step by step.
A: Each machine makes 1 widget in 5 minutes.
   100 machines each make 1 widget in 5 minutes.
   So 100 machines make 100 widgets in 5 minutes.
   Answer: 5   ← correct
```

### Types of CoT

| Type | Description | When to use |
|---|---|---|
| **Zero-shot CoT** | Just add "Think step by step" to the prompt | Quick baseline |
| **Few-shot CoT** | Provide 2-4 worked examples in the prompt | Better accuracy |
| **Self-consistency** | Sample CoT N times, majority vote | Best accuracy |

In this notebook we explore zero-shot and few-shot CoT. Self-consistency comes in notebook 3.

---

## Setup

### Model choice: `Qwen/Qwen2.5-Math-7B-Instruct`

**Why this model?**
- **Math-specialized:** Qwen2.5-Math was pretrained on large math corpora (textbooks, papers, competition problems), then instruction-tuned. It dramatically outperforms general-purpose 7B models on math benchmarks.
- **7B parameters:** Fits on a single T4 GPU (free Kaggle/Colab tier) with 4-bit quantization, or comfortably on an A10G/A100.
- **Open weights:** Available on HuggingFace under an open license.
- **Strong baseline:** Scores ~75% on the MATH benchmark vs ~50% for general models of the same size.

### Hardware requirements

| Platform | Backend | Quantization | Notes |
|---|---|---|---|
| **Apple Silicon Mac** (M1/M2/M3/M4) | MLX + Metal | 4-bit via mlx-lm | Fastest local option — uses unified memory |
| **Kaggle / Colab T4 (16GB)** | CUDA + bitsandbytes | NF4 4-bit | Free tier, ~6GB VRAM used |
| **A10G / A100 / P100** | CUDA | NF4 4-bit | Comfortable headroom |
| **CPU-only** | — | — | Very slow, not recommended |

The code below automatically detects the platform and uses the right backend.

### Install dependencies

In [1]:
import platform, subprocess, sys

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    # Apple Silicon: mlx-lm runs natively on Metal (GPU/Neural Engine), no CUDA needed.
    # bitsandbytes is CUDA-only and will NOT install on Mac — use mlx-lm instead.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlx-lm"], check=True)
    print("Installed: mlx-lm  (Apple Silicon / Metal path)")
else:
    # Kaggle / Colab / Linux CUDA path
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "bitsandbytes", "accelerate"],
        check=True,
    )
    print("Installed: transformers + bitsandbytes  (CUDA / Kaggle path)")

print(f"Platform detected: {'macOS (Apple Silicon)' if IS_MAC else 'CUDA / other'}")

Installed: mlx-lm  (Apple Silicon / Metal path)
Platform detected: macOS (Apple Silicon)


### Load the model

**On Apple Silicon (macOS):**  
We use `mlx-lm` which loads a pre-quantized 4-bit MLX model (`mlx-community/Qwen2.5-Math-7B-Instruct-4bit`).  
MLX runs on the Metal GPU/Neural Engine — no CUDA, no bitsandbytes needed.  
The 7B model fits comfortably in 16GB unified memory (typical M-series Mac).

**On CUDA (Kaggle / Colab):**  
We use `bitsandbytes` NF4 4-bit quantization via `transformers`.  
This compresses the 7B model from ~14GB → ~4.5GB VRAM, fitting on a free T4.

In [2]:
import platform

IS_MAC = platform.system() == "Darwin"
MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"

if IS_MAC:
    # ── Apple Silicon path ────────────────────────────────────────────────────
    # mlx-lm's load() returns an MLX model + a HuggingFace-compatible tokenizer.
    # The 4-bit MLX variant is hosted by the mlx-community org on HuggingFace.
    from mlx_lm import load, generate as _mlx_generate
    model, tokenizer = load("mlx-community/Qwen2.5-Math-7B-Instruct-4bit")
    print("Model loaded on Apple Silicon (MLX + Metal)")

else:
    # ── CUDA path (Kaggle / Colab) ────────────────────────────────────────────
    # 4-bit quantization config:
    #   load_in_4bit          → compress weights from 32-bit to 4-bit
    #   bnb_4bit_compute_dtype → use bfloat16 for the actual matrix multiplications
    #   bnb_4bit_use_double_quant → quantize the quant constants too (saves a bit more)
    #   bnb_4bit_quant_type   → NF4 is the recommended quantization format
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print(f"Model loaded on CUDA. Device: {next(model.parameters()).device}")

/Users/pulin05/kaggling/kaggling/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [07:18<00:00, 48.70s/it] 


Model loaded on Apple Silicon (MLX + Metal)


### Define a simple inference helper

Key parameters to understand:
- **`max_new_tokens`:** How many tokens the model is allowed to generate. Math solutions can be long — 1024 is a good starting point.
- **`temperature`:** Controls randomness. `temperature=0` → deterministic (always picks the most likely token). `temperature=0.7` → some randomness, useful for sampling diverse solutions later.
- **`do_sample`:** Must be `True` to use temperature > 0. If `False`, uses greedy decoding.

In [19]:
def generate_response(prompt: str, max_new_tokens: int = 1024, temperature: float = 0.0) -> str:
    """
    Send a prompt to the model and return the generated text.

    Supports two backends selected at load time:
      - IS_MAC=True  -> mlx_lm.generate  (Apple Silicon / Metal)
      - IS_MAC=False -> transformers model.generate  (CUDA / Kaggle)

    Both paths use apply_chat_template so the instruction format is identical.
    """
    messages = [{"role": "user", "content": prompt}]

    # apply_chat_template formats the messages into the token sequence
    # the model was trained to expect (e.g., <|im_start|>user\n...)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # adds the assistant turn opener
    )

    if IS_MAC:
        # mlx-lm 0.21+ API: temperature is passed via a sampler callable,
        # not as a direct kwarg to generate().
        from mlx_lm.generate import make_sampler
        return _mlx_generate(
            model, tokenizer,
            prompt=text,
            max_tokens=max_new_tokens,
            sampler=make_sampler(temp=temperature),
            verbose=False,
        )
    else:
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():  # don't track gradients during inference
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0),
                temperature=temperature if temperature > 0 else None,
                pad_token_id=tokenizer.eos_token_id,
            )
        # Decode only the newly generated tokens (not the input prompt)
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(generated_ids, skip_special_tokens=True)


---

## Load the Reference Benchmark

We use the official `reference.csv` — 10 real AIMO3 problems with verified answers.  
Problems 1–4 come from AIMO2. Problems 5–10 are novel AIMO3-level problems.

**Expected model scores on these 10 problems (from the PDF):**
- GPT-5 Pro: 10/10
- Gemini 2.5 Pro: 9/10  
- Best open model (gpt-oss-120b, 120B params): **4/10** — only Problems 1–4  
- Qwen2.5-Math-7B (our model): expect **1–3/10** — a strong result at 7B scale

This gives you a reality check: if you get Problems 1–2 correct, your pipeline is working.  
Problems 5–10 are genuinely unsolved by all open models under ~100B parameters.

In [20]:
import pandas as pd

# Load the official reference benchmark
# When running on Kaggle, change this path to:
# '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv'
REFERENCE_CSV = '../Dataset/reference.csv'

ref_df = pd.read_csv(REFERENCE_CSV)
print(f"Loaded {len(ref_df)} reference problems.\n")
print(ref_df[['id', 'answer']].to_string(index=False))

# Convert to list-of-dicts for easy iteration (same interface as before)
problems = []
for _, row in ref_df.iterrows():
    problems.append({
        "id":      row['id'],
        "problem": row['problem'],
        "answer":  int(row['answer']),
    })

# Annotate difficulty tiers based on the PDF
# Problems 1-4: AIMO2 carry-overs (solvable by most models)
# Problems 5-10: Novel AIMO3-level (currently only frontier closed models solve these)
difficulty = {
    '92ba6a': ('P1', 'AIMO2', 'Easy'),
    'a295e9': ('P2', 'AIMO2', 'Medium'),
    '0e644e': ('P3', 'AIMO2', 'Hard'),
    '9c1c5f': ('P4', 'AIMO2', 'Very Hard'),
    '424e18': ('P5', 'AIMO3', 'Extreme'),
    '26de63': ('P6', 'AIMO3', 'Extreme'),
    '42d360': ('P7', 'AIMO3', 'Extreme'),
    '641659': ('P8', 'AIMO3', 'Extreme'),
    'dd7f5e': ('P9', 'AIMO3', 'Extreme'),
    '86e8e5': ('P10', 'AIMO3', 'Extreme'),
}

print("\nDifficulty breakdown:")
for p in problems:
    d = difficulty.get(p['id'], ('?', '?', '?'))
    print(f"  {d[0]:4s} [{d[1]:5s}] {d[2]:9s} | id={p['id']} | answer={p['answer']}")

Loaded 10 reference problems.

    id  answer
0e644e     336
26de63   32951
424e18   21818
42d360   32193
641659   57447
86e8e5    8687
92ba6a      50
9c1c5f     580
a295e9     520
dd7f5e     160

Difficulty breakdown:
  P3   [AIMO2] Hard      | id=0e644e | answer=336
  P6   [AIMO3] Extreme   | id=26de63 | answer=32951
  P5   [AIMO3] Extreme   | id=424e18 | answer=21818
  P7   [AIMO3] Extreme   | id=42d360 | answer=32193
  P8   [AIMO3] Extreme   | id=641659 | answer=57447
  P10  [AIMO3] Extreme   | id=86e8e5 | answer=8687
  P1   [AIMO2] Easy      | id=92ba6a | answer=50
  P4   [AIMO2] Very Hard | id=9c1c5f | answer=580
  P2   [AIMO2] Medium    | id=a295e9 | answer=520
  P9   [AIMO3] Extreme   | id=dd7f5e | answer=160


---

## Experiment 1: Baseline (No CoT)

We ask the model to answer directly, with no instruction to show reasoning. This is the simplest possible prompt.

**Expected outcome:** The model will get some AMC problems right by pattern matching, but will fail on harder ones. It has no mechanism to catch its own arithmetic errors.

In [21]:
def make_baseline_prompt(problem_text: str) -> str:
    return (
        f"Solve this math problem. Your answer must be an integer.\n\n"
        f"Problem: {problem_text}\n\n"
        f"Answer: "
    )

def extract_integer_answer(text: str) -> int | None:
    """
    Extract the last integer found in the model output.
    Models often embed the answer in a sentence; we grab the last number.
    """
    import re
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    if not numbers:
        return None
    return int(numbers[-1])


print("=" * 60)
print("EXPERIMENT 1: BASELINE (NO COT)")
print("=" * 60)

baseline_results = []

for p in problems:
    d = difficulty.get(p['id'], ('?', '?', '?'))   # (tier, source, level)
    prompt    = make_baseline_prompt(p["problem"])
    response  = generate_response(prompt, max_new_tokens=256, temperature=0.0)
    predicted = extract_integer_answer(response)
    correct   = (predicted == p["answer"])

    baseline_results.append({
        "id":        p["id"],
        "tier":      d[0],
        "source":    d[1],
        "difficulty":d[2],
        "expected":  p["answer"],
        "predicted": predicted,
        "correct":   correct,
        "response":  response,
    })

    status = "CORRECT" if correct else "WRONG"
    print(f"[{d[0]:3s}] {d[2]:9s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print(f"       Response snippet: {response[:120].strip()}")
    print()

n_correct = sum(r["correct"] for r in baseline_results)
print(f"\nBaseline score: {n_correct}/{len(problems)}")

EXPERIMENT 1: BASELINE (NO COT)
[P3 ] Hard      | Expected:    336 | Got: None   | WRONG
       Response snippet: Given the problem, we start by analyzing the geometric configuration and the conditions provided. We know that \(AD = AE

[P6 ] Extreme   | Expected:  32951 | Got: 1      | WRONG
       Response snippet: To solve the problem, we need to analyze the function \( f(n) \) and determine the value of \( N = f(M^{15}) - f(M^{15}

[P5 ] Extreme   | Expected:  21818 | Got: 1      | WRONG
       Response snippet: To solve this problem, we need to determine the number of possible orderings of the competitors at the end of the tourna

[P7 ] Extreme   | Expected:  32193 | Got: 1      | WRONG
       Response snippet: To determine the largest possible number of moves Ken could make, we need to understand the process he follows. Ken star

[P8 ] Extreme   | Expected:  57447 | Got: 1      | WRONG
       Response snippet: To solve the problem, we need to analyze the given geometric configurat

---

## Experiment 2: Zero-Shot CoT

We add **"Think step by step before giving the final answer."** to the prompt. That's the only change. We also change the output format to make it easier to extract the final answer.

**Expected outcome:** Noticeably better, especially on the combinatorics and number theory problems where multiple steps are required.

In [22]:
def make_zeroshot_cot_prompt(problem_text: str) -> str:
    return (
        f"Solve this math problem. Think step by step before giving the final answer.\n"
        f"At the end, write your final integer answer on a new line as:\n"
        f"ANSWER: <integer>\n\n"
        f"Problem: {problem_text}\n\n"
        f"Solution:"
    )

def extract_labeled_answer(text: str) -> int | None:
    """
    Look for 'ANSWER: <number>' in the model's output.
    Fall back to extracting the last integer if the label is missing.
    """
    import re
    match = re.search(r"ANSWER:\s*(-?\d+)", text, re.IGNORECASE)
    if match:
        return int(match.group(1))
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    return int(numbers[-1]) if numbers else None


print("=" * 60)
print("EXPERIMENT 2: ZERO-SHOT COT")
print("=" * 60)

zs_cot_results = []

for p in problems:
    d = difficulty.get(p['id'], ('?', '?', '?'))
    prompt    = make_zeroshot_cot_prompt(p["problem"])
    response  = generate_response(prompt, max_new_tokens=1024, temperature=0.0)
    predicted = extract_labeled_answer(response)
    correct   = (predicted == p["answer"])

    zs_cot_results.append({
        "id":        p["id"],
        "tier":      d[0],
        "source":    d[1],
        "difficulty":d[2],
        "expected":  p["answer"],
        "predicted": predicted,
        "correct":   correct,
        "response":  response,
    })

    status = "CORRECT" if correct else "WRONG"
    print(f"[{d[0]:3s}] {d[2]:9s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print()

n_correct = sum(r["correct"] for r in zs_cot_results)
print(f"\nZero-shot CoT score: {n_correct}/{len(problems)}")

EXPERIMENT 2: ZERO-SHOT COT
[P3 ] Hard      | Expected:    336 | Got: 120    | WRONG

[P6 ] Extreme   | Expected:  32951 | Got: 78125  | WRONG

[P5 ] Extreme   | Expected:  21818 | Got: 13     | WRONG

[P7 ] Extreme   | Expected:  32193 | Got: 32192  | WRONG

[P8 ] Extreme   | Expected:  57447 | Got: 1      | WRONG

[P10] Extreme   | Expected:   8687 | Got: 2025   | WRONG

[P1 ] Easy      | Expected:     50 | Got: 4      | WRONG

[P4 ] Very Hard | Expected:    580 | Got: 1      | WRONG

[P2 ] Medium    | Expected:    520 | Got: 500    | WRONG

[P9 ] Extreme   | Expected:    160 | Got: 36     | WRONG


Zero-shot CoT score: 0/10


---

## Experiment 3: Few-Shot CoT

We provide **2 worked examples** in the prompt before the actual question. This is called "in-context learning" — the model observes the pattern (problem → step-by-step solution → ANSWER) and applies it.

**Why does this help beyond zero-shot CoT?**  
The examples anchor the *style* of reasoning. For math, they show: how to set up equations, how to name variables, how to structure the work. The model's generation is then constrained to follow that style.

**Important:** We use examples from **different topics** than the test problem, so we're not just showing the answer — we're showing the *process*.

In [23]:
FEW_SHOT_EXAMPLES = """\
Problem: A train travels 120 km in 2 hours, then 180 km in 3 hours. What is its average speed over the entire journey?
Solution:
Step 1: Total distance = 120 + 180 = 300 km
Step 2: Total time = 2 + 3 = 5 hours
Step 3: Average speed = total distance / total time = 300 / 5 = 60 km/h
ANSWER: 60

Problem: How many 3-digit numbers have all distinct digits?
Solution:
Step 1: The hundreds digit can be 1–9 (not 0): 9 choices.
Step 2: The tens digit can be 0–9 except the hundreds digit: 9 choices.
Step 3: The units digit can be 0–9 except the two already chosen: 8 choices.
Step 4: Total = 9 × 9 × 8 = 648.
ANSWER: 648
"""

def make_fewshot_cot_prompt(problem_text: str) -> str:
    return (
        f"Solve math problems step by step. Write ANSWER: <integer> at the end.\n\n"
        f"{FEW_SHOT_EXAMPLES}\n"
        f"Problem: {problem_text}\n"
        f"Solution:"
    )


print("=" * 60)
print("EXPERIMENT 3: FEW-SHOT COT")
print("=" * 60)

fs_cot_results = []

for p in problems:
    d = difficulty.get(p['id'], ('?', '?', '?'))
    prompt    = make_fewshot_cot_prompt(p["problem"])
    response  = generate_response(prompt, max_new_tokens=1024, temperature=0.0)
    predicted = extract_labeled_answer(response)
    correct   = (predicted == p["answer"])

    fs_cot_results.append({
        "id":        p["id"],
        "tier":      d[0],
        "source":    d[1],
        "difficulty":d[2],
        "expected":  p["answer"],
        "predicted": predicted,
        "correct":   correct,
        "response":  response,
    })

    status = "CORRECT" if correct else "WRONG"
    print(f"[{d[0]:3s}] {d[2]:9s} | Expected: {p['answer']:6d} | Got: {str(predicted):6s} | {status}")
    print()

n_correct = sum(r["correct"] for r in fs_cot_results)
print(f"\nFew-shot CoT score: {n_correct}/{len(problems)}")

EXPERIMENT 3: FEW-SHOT COT
[P3 ] Hard      | Expected:    336 | Got: 2730   | WRONG

[P6 ] Extreme   | Expected:  32951 | Got: 0      | WRONG

[P5 ] Extreme   | Expected:  21818 | Got: 20     | WRONG

[P7 ] Extreme   | Expected:  32193 | Got: 32193  | CORRECT

[P8 ] Extreme   | Expected:  57447 | Got: 57417  | WRONG

[P10] Extreme   | Expected:   8687 | Got: 7      | WRONG

[P1 ] Easy      | Expected:     50 | Got: 50     | CORRECT

[P4 ] Very Hard | Expected:    580 | Got: 1      | WRONG

[P2 ] Medium    | Expected:    520 | Got: 999    | WRONG

[P9 ] Extreme   | Expected:    160 | Got: 36     | WRONG


Few-shot CoT score: 2/10


---

## Results Comparison

In [24]:
print("=" * 80)
print(f"{'#':<5} {'Tier':<8} {'Expected':<10} {'Baseline':<10} {'ZS-CoT':<10} {'FS-CoT':<10} {'Answer'}")
print("-" * 80)

for i, p in enumerate(problems):
    d = difficulty.get(p['id'], ('?', '?', '?'))
    b = "✓" if baseline_results[i]["correct"] else "✗"
    z = "✓" if zs_cot_results[i]["correct"] else "✗"
    f = "✓" if fs_cot_results[i]["correct"] else "✗"
    tier = f"{d[0]} {d[2][:3]}"
    print(f"{i+1:<5} {tier:<8} {p['answer']:<10} {b:<10} {z:<10} {f:<10}")

print("-" * 80)
bl = sum(r["correct"] for r in baseline_results)
zs = sum(r["correct"] for r in zs_cot_results)
fs = sum(r["correct"] for r in fs_cot_results)
n  = len(problems)
print(f"{'TOTAL':<24} {bl}/{n}      {zs}/{n}      {fs}/{n}")
print("=" * 80)
print()
print("Context: best open-weight model (120B params) scores 4/10 on this benchmark.")
print("         Scoring 1-2/10 with a 7B model via CoT is expected and a good start.")
print("         The path to higher scores is: TIR loop (notebook 02) + majority voting (03).")

#     Tier     Expected   Baseline   ZS-CoT     FS-CoT     Answer
--------------------------------------------------------------------------------
1     P3 Har   336        ✗          ✗          ✗         
2     P6 Ext   32951      ✗          ✗          ✗         
3     P5 Ext   21818      ✗          ✗          ✗         
4     P7 Ext   32193      ✗          ✗          ✓         
5     P8 Ext   57447      ✗          ✗          ✗         
6     P10 Ext  8687       ✗          ✗          ✗         
7     P1 Eas   50         ✗          ✗          ✓         
8     P4 Ver   580        ✗          ✗          ✗         
9     P2 Med   520        ✗          ✗          ✗         
10    P9 Ext   160        ✗          ✗          ✗         
--------------------------------------------------------------------------------
TOTAL                    0/10      0/10      2/10

Context: best open-weight model (120B params) scores 4/10 on this benchmark.
         Scoring 1-2/10 with a 7B model via CoT is exp

---

## Inspect the Failures

This is the most important part. For every problem where the model failed, read the full response and understand *why* it went wrong. This directly informs what we build next.

In [25]:
print("FAILURE ANALYSIS — Few-Shot CoT Errors")
print("=" * 60)

for i, p in enumerate(problems):
    r = fs_cot_results[i]
    if not r["correct"]:
        print(f"\nProblem {r['tier']} ({r['difficulty']}):")
        print(f"  Problem : {p['problem'][:200]}...")
        print(f"  Expected: {p['answer']}")
        print(f"  Got     : {r['predicted']}")
        print(f"  Model output:")
        print("  " + r["response"].replace("\n", "\n  "))
        print()

failures = [r for r in fs_cot_results if not r["correct"]]
if not failures:
    print("No failures! Move to notebook 02 (TIR loop).")

FAILURE ANALYSIS — Few-Shot CoT Errors

Problem P3 (Hard):
  Problem : Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Cir...
  Expected: 336
  Got     : 2730
  Model output:
  To solve the problem, we need to find the unique triangle with integer side lengths \(a = BC\), \(b = CA\), and \(c = AB\) such that \(AB < AC\) and the conditions given in the problem are satisfied. We need to find the remainder when \(abc\) is divided by \(10^5\).
  
  First, let's analyze the given conditions. We have a triangle \(ABC\) with \(AB < AC\) and points \(D\) and \(E\) on \(BC\) and \(AC\) respectively, such that \(AD = AE = AB = c\). Line \(DE\) intersects \(AB\) at \(X\), and circles \(BXD\) and \(CED\) intersect for the second time at \(Y \neq D\), with \(Y\) lying on line \(AD\).
  
  Since \(AD = AB = c\) and \(AE = AB = c\), triangle \(ADE\) is i

---

## What Did We Learn?

### Key observations (fill in after running)

1. **Baseline vs CoT gap:** CoT improved accuracy by ___ problems. The reasoning traces make the difference visible.

2. **The AIMO2 vs AIMO3 divide:** Problems P1–P4 (AIMO2 carry-overs) are where CoT has a chance.  
   P5–P10 (novel AIMO3 problems) are currently out of reach for text-only reasoning regardless of prompt engineering — they require either 100B+ parameters or the code execution loop we build next.

3. **Where CoT still fails even on easier problems:**
   - Arithmetic errors over many steps (no external calculator)
   - Combinatorics with many cases (misses cases or double-counts)
   - Modular arithmetic / Legendre's formula (needs precise integer computation)

4. **What the failure reveals:** When you read the model's failed reasoning trace for P3 (the acute triangle) or P4 (functional equation), you'll see it *starts* correctly — sets up the right framework — then makes an algebraic slip. This is exactly the gap that code execution closes.

### What comes next (Notebook 02)

**Tool-Integrated Reasoning (TIR):** teach the model to interleave Python code with its reasoning.  
Instead of computing `(x+y)^2 - 2xy` mentally, it writes:
```python
from sympy import symbols, expand
x, y = symbols('x y')
# x+y=10, xy=20 → x^2+y^2 = (x+y)^2 - 2xy
result = 10**2 - 2*20
print(result)  # 60
```
Python computes exactly. No hallucination. The model reads the output and continues.

This is the single biggest accuracy jump in the AIMO progression.

---

## Exercises (before moving to notebook 02)

1. **Read the reference PDF** (`Dataset/AIMO3_Reference_Problems.pdf`) — skim the solutions for  
   P1 (sweets/ages) and P2 (rectangle tiling). These are the problems your model is most likely  
   to solve. Understanding the correct approach helps you evaluate whether the model's CoT is  
   on the right track even when it gets the final answer wrong.

2. **Try temperature > 0** — re-run P1 five times with `temperature=0.7`.  
   How often does the model get it right? This variance is *exactly* why majority voting (Phase 3) matters.

3. **Inspect the tokenization of a problem** — run:
   ```python
   tokens = tokenizer.encode(problems[0]['problem'])
   print(tokenizer.convert_ids_to_tokens(tokens))
   ```
   Notice how `$x + y = 10$` is tokenized. LaTeX math tokens are often split in non-intuitive ways.

4. **Read the failure for P4** (functional equation, ~2% human solve rate at AIMO2).  
   This problem requires recognizing that `f(m+n+mn) = f((m+1)(n+1)-1)` maps to a multiplicative  
   structure. Does the model *see* this substitution? If it does but then fails, the failure is  
   computational. If it doesn't, the failure is conceptual — a different kind of problem.